In [1]:
%pip install --no-cache-dir --force-reinstall "numpy==1.26.4" "pyarrow==17.0.0" "datasets==3.6.0" "opencv-python==4.11.0.86" "mediapipe==0.10.21" scikit-learn pandas matplotlib joblib pillow
print("Installation complete. Restart the Jupyter kernel before running the next cell.")

INFO: pip is looking at multiple versions of pandas to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opencv-contrib-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 1.7 MB/s  0:00:08 eta 0:00:01
   ━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/27.2 MB 1.6 MB/s eta 0:00:16
Resuming download pyarrow-17.0.0-cp311-cp311-macosx_11_0_arm64.whl (1.0 MB/27.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.2/27.2 MB 3.3 MB/s  0:00:08m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 2.7 MB/s  0:00:13m0:

In [39]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import joblib
import math
import time
import random
import io
from pathlib import Path
from collections import deque, Counter
from PIL import Image
from datasets import load_dataset, Image as HFImage
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score
import matplotlib.pyplot as plt

In [40]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [41]:
CHARACTERS = {
    "1": "WANDA",
    "2": "DOCTOR STRANGE",
    "3": "THOR",
    "4": "LOKI",
    "5": "FIRE MAGIC",
    "6": "PURPLE COSMIC",
    "7": "SHANG-CHI",
    "8": "THANOS",
    "9": "ANCIENT ONE",
}
PALETTES = {
    "WANDA": (45, 35, 245),
    "DOCTOR STRANGE": (35, 155, 255),
    "THOR": (255, 225, 100),
    "LOKI": (70, 220, 95),
    "FIRE MAGIC": (25, 125, 255),
    "PURPLE COSMIC": (220, 70, 205),
    "SHANG-CHI": (30, 195, 255),
    "THANOS": (190, 65, 165),
    "ANCIENT ONE": (185, 220, 80),
}

In [42]:
mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

In [43]:
print("OpenCV:", cv2.__version__, "| MediaPipe:", mp.__version__, "| Dataset:", DATASET_ID)

OpenCV: 4.11.0 | MediaPipe: 0.10.21 | Dataset: ntsrigaud/hagrid-subset


In [44]:
import sys
import numpy as np
import pyarrow as pa
import datasets
print("Python:", sys.executable)
print("NumPy:", np.__version__)
print("PyArrow:", pa.__version__)
print("Hugging Face Datasets:", datasets.__version__)
assert np.__version__.startswith("1.26."), "Wrong NumPy version: restart kernel and verify the active environment."
assert pa.__version__.startswith("17.0."), "Wrong PyArrow version: reinstall in this kernel environment."
print("NumPy/PyArrow compatibility imports passed.")

Python: /opt/anaconda3/envs/marvel-magic/bin/python
NumPy: 1.26.4
PyArrow: 17.0.0
Hugging Face Datasets: 3.6.0
NumPy/PyArrow compatibility imports passed.


In [45]:
def landmark_array(hand_landmarks):
    return np.asarray([(p.x, p.y, p.z) for p in hand_landmarks.landmark], dtype=np.float32)

In [46]:
def normalized_features(points):
    points = np.asarray(points, dtype=np.float32).copy()
    if points.shape != (21, 3) or not np.isfinite(points).all():
        return None
    points -= points[0]
    scale = np.max(np.linalg.norm(points[:, :2], axis=1))
    if not np.isfinite(scale) or scale < 1e-6:
        return None
    points /= scale
    return points.reshape(-1)

In [47]:
def pixel_point(points, index, width, height):
    return (int(np.clip(points[index, 0] * width, 0, width - 1)),
            int(np.clip(points[index, 1] * height, 0, height - 1)))

In [48]:
def distance2d(a, b):
    return float(np.linalg.norm(np.asarray(a[:2]) - np.asarray(b[:2])))

In [49]:
def finger_states(points):
    states = []
    for tip, pip, mcp in ((8, 6, 5), (12, 10, 9), (16, 14, 13), (20, 18, 17)):
        states.append(bool(distance2d(points[tip], points[0]) > distance2d(points[mcp], points[0]) * 1.18))
    thumb_extended = distance2d(points[4], points[5]) > distance2d(points[3], points[5]) * 1.08
    return [thumb_extended] + states

In [50]:
class HandTracker:
    def __init__(self, max_hands=2, static_image_mode=False):
        self.detector = mp_hands.Hands(
            static_image_mode=static_image_mode, max_num_hands=max_hands,
            model_complexity=1, min_detection_confidence=0.55,
            min_tracking_confidence=0.55
        )

    def process(self, frame):
        result = self.detector.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        detected = []
        if result.multi_hand_landmarks:
            for index, landmarks in enumerate(result.multi_hand_landmarks):
                handedness = "Unknown"
                if result.multi_handedness and index < len(result.multi_handedness):
                    handedness = result.multi_handedness[index].classification[0].label
                if handedness == "Unknown":
                    handedness = f"Unknown-{index}"
                detected.append({"points": landmark_array(landmarks), "handedness": handedness,
                                 "landmarks": landmarks})
        return detected

    def close(self):
        self.detector.close()

print("Landmark extraction and live tracking are ready.")

Landmark extraction and live tracking are ready.


In [51]:
# LOCAL DATASET CONFIGURATION — no downloading / Hugging Face calls.
# Expected GitHub repository layout:
# HandGesture-Recognizer/
#   dataset/
#     OpenPalm_1.jpg ... OpenPalm_50.jpg
#     PeaceSign_1.jpg ... PeaceSign_50.jpg
#     Fist_1.jpg ... Fist_50.jpg
#     ThumbsUp_1.jpg ... ThumbsUp_50.jpg

PROJECT_DIR = Path.cwd()
DATASET_DIR = PROJECT_DIR / "HandGesture-Recognizer" / "dataset"

# If the dataset folder is directly inside your current project, use:
# DATASET_DIR = PROJECT_DIR / "dataset"

# If you keep the repository somewhere else, replace the line above with its exact path, e.g.:
# DATASET_DIR = Path("/Users/your-user-name/path/to/HandGesture-Recognizer/dataset")

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
SEED = 42
MAX_IMAGES_PER_CLASS = None  # None = use every matching image

# The downloaded repository has OpenPalm, PeaceSign, Fist and ThumbsUp.
# Only OpenPalm and PeaceSign have supervised labels in this model.
# POINT is recognized by a separate MediaPipe landmark rule at live time.
FILE_LABELS = {
    "openpalm": "OPEN_PALM",
    "peacesign": "TWO_FINGERS",
    "fist": None,       # intentionally excluded; not one of the requested classes
    "thumbsup": None,   # intentionally excluded; not one of the requested classes
}

POSE_LABELS = ["OPEN_PALM", "POINT", "TWO_FINGERS"]
TRAINED_LABELS = ["OPEN_PALM", "TWO_FINGERS"]

BASE_DIR = PROJECT_DIR / "mystic_motion_project"
FEATURE_CACHE = BASE_DIR / "local_hand_features.csv"
MODEL_PATH = BASE_DIR / "mystic_motion_gesture_model.joblib"
BASE_DIR.mkdir(parents=True, exist_ok=True)

if not DATASET_DIR.exists():
    candidates = [
        PROJECT_DIR / "dataset",
        PROJECT_DIR / "HandGesture-Recognizer" / "dataset",
        PROJECT_DIR.parent / "HandGesture-Recognizer" / "dataset",
    ]
    found = [p for p in candidates if p.exists()]
    if found:
        DATASET_DIR = found[0]

print("Project directory:", PROJECT_DIR.resolve())
print("Dataset directory:", DATASET_DIR.resolve())
print("Dataset folder exists:", DATASET_DIR.exists())

Project directory: /Users/varshh06/Desktop/CV-END-TO-END/03. ARTIFICIAL MAGIC SYTSTEM
Dataset directory: /Users/varshh06/Desktop/CV-END-TO-END/03. ARTIFICIAL MAGIC SYTSTEM/dataset
Dataset folder exists: True


In [78]:
from pathlib import Path

FILE_LABELS = {
    "openpalm": "OPEN_PALM",
    "peacesign": "TWO_FINGERS",
    "fist": None,
    "thumbsup": None,
}

def label_from_filename(path):
    filename = Path(path).stem.lower()

    for prefix, label in FILE_LABELS.items():
        if filename.startswith(prefix):
            return label

    return None

In [79]:
test_files = [
    "OpenPalm_1.jpg",
    "PeaceSign_1.jpg",
    "Fist_1.jpg",
    "ThumbsUp_1.jpg"
]

for file in test_files:
    print(file, "→", label_from_filename(file))

OpenPalm_1.jpg → OPEN_PALM
PeaceSign_1.jpg → TWO_FINGERS
Fist_1.jpg → None
ThumbsUp_1.jpg → None


In [52]:
def source_label_for_path(path):
    label, raw = label_from_filename(path)
    return label, raw

In [53]:
print("Supervised classes:", TRAINED_LABELS)
print("Live gesture set:", POSE_LABELS)
print("POINT will be detected by a transparent landmark rule, not claimed as a trained class.")

Supervised classes: ['OPEN_PALM', 'TWO_FINGERS']
Live gesture set: ['OPEN_PALM', 'POINT', 'TWO_FINGERS']
POINT will be detected by a transparent landmark rule, not claimed as a trained class.


In [54]:
def read_bgr_image(path):
    image = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError(f"Could not decode image: {path}")
    return image

In [83]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
DATASET_DIR = PROJECT_DIR / "dataset"  # Singular, not datasets

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

if not DATASET_DIR.exists():
    raise FileNotFoundError(
        f"Dataset folder not found: {DATASET_DIR.resolve()}"
    )

all_image_paths = sorted(
    p for p in DATASET_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
)

print("Dataset location:", DATASET_DIR.resolve())
print("Total images found:", len(all_image_paths))

if not all_image_paths:
    raise ValueError("No images found inside the dataset folder!")

print("\nSample image paths:")
for path in all_image_paths[:10]:
    print(path)

Dataset location: /Users/varshh06/Desktop/CV-END-TO-END/03. ARTIFICIAL MAGIC SYTSTEM/dataset
Total images found: 200

Sample image paths:
/Users/varshh06/Desktop/CV-END-TO-END/03. ARTIFICIAL MAGIC SYTSTEM/dataset/Fist_1.jpg
/Users/varshh06/Desktop/CV-END-TO-END/03. ARTIFICIAL MAGIC SYTSTEM/dataset/Fist_10.jpg
/Users/varshh06/Desktop/CV-END-TO-END/03. ARTIFICIAL MAGIC SYTSTEM/dataset/Fist_11.jpg
/Users/varshh06/Desktop/CV-END-TO-END/03. ARTIFICIAL MAGIC SYTSTEM/dataset/Fist_12.jpg
/Users/varshh06/Desktop/CV-END-TO-END/03. ARTIFICIAL MAGIC SYTSTEM/dataset/Fist_13.jpg
/Users/varshh06/Desktop/CV-END-TO-END/03. ARTIFICIAL MAGIC SYTSTEM/dataset/Fist_14.jpg
/Users/varshh06/Desktop/CV-END-TO-END/03. ARTIFICIAL MAGIC SYTSTEM/dataset/Fist_15.jpg
/Users/varshh06/Desktop/CV-END-TO-END/03. ARTIFICIAL MAGIC SYTSTEM/dataset/Fist_16.jpg
/Users/varshh06/Desktop/CV-END-TO-END/03. ARTIFICIAL MAGIC SYTSTEM/dataset/Fist_17.jpg
/Users/varshh06/Desktop/CV-END-TO-END/03. ARTIFICIAL MAGIC SYTSTEM/dataset/Fist_

In [86]:
LANDMARK_COUNT = 21
COORDINATES = ["x", "y", "z"]

FEATURE_COLUMNS = [
    f"landmark_{i}_{axis}"
    for i in range(LANDMARK_COUNT)
    for axis in COORDINATES
]

print("Total feature columns:", len(FEATURE_COLUMNS))
print("Feature names:", FEATURE_COLUMNS[:6])

Total feature columns: 63
Feature names: ['landmark_0_x', 'landmark_0_y', 'landmark_0_z', 'landmark_1_x', 'landmark_1_y', 'landmark_1_z']


In [87]:
observed_labels = Counter()
mapped_label_counts = Counter()
excluded_label_counts = Counter()
failed_images = 0
failed_detections = 0
feature_rows = []

image_tracker = HandTracker(max_hands=2, static_image_mode=True)
try:
    for image_path in all_image_paths:
        target, raw_name = source_label_for_path(image_path)
        observed_labels[raw_name] += 1

        if target is None:
            excluded_label_counts[raw_name] += 1
            continue

        if MAX_IMAGES_PER_CLASS is not None and mapped_label_counts[target] >= MAX_IMAGES_PER_CLASS:
            continue

        try:
            bgr = read_bgr_image(image_path)
            detected_hands = image_tracker.process(bgr)
            if not detected_hands:
                failed_detections += 1
                continue

            hand = detected_hands[0]
            vector = normalized_features(hand["points"])
            if len(vector) != len(FEATURE_COLUMNS):
                raise ValueError(
                    f"Feature mismatch: got {len(vector)} values, "
                    f"expected {len(FEATURE_COLUMNS)}"
                )
            if vector is None:
                failed_detections += 1
                continue

            record = {
                "label": target,
                "source_split": "local",
                "source_group": image_path.name,
                "source_label": raw_name,
                "source_path": str(image_path),
            }
            record.update(dict(zip(FEATURE_COLUMNS, vector.astype(float))))
            feature_rows.append(record)
            mapped_label_counts[target] += 1

        except Exception as error:
            failed_images += 1
            print("Image processing error:", image_path.name, "|", error)
finally:
    image_tracker.close()

print("Feature extraction complete.")

I0000 00:00:1793957071.204587  927830 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M3
W0000 00:00:1793957071.218681 1026434 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1793957071.227749 1026432 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


TypeError: cannot unpack non-iterable NoneType object

In [56]:
if not feature_rows:
    raise RuntimeError("No usable hand landmarks were extracted. Check the dataset path and image quality.")

pose_df = pd.DataFrame(feature_rows)
pose_df.to_csv(FEATURE_CACHE, index=False)
print("Feature cache saved:", FEATURE_CACHE.resolve())
print(pose_df["label"].value_counts())

RuntimeError: No usable hand landmarks were extracted. Check the dataset path and image quality.

In [57]:
pose_df = pd.read_csv(FEATURE_CACHE)
required = ["label", "source_split", "source_group"] + FEATURE_COLUMNS
missing = [column for column in required if column not in pose_df.columns]
if missing:
    raise ValueError(f"Feature cache missing columns: {missing}")
pose_df = pose_df.dropna(subset=required).copy()

FileNotFoundError: [Errno 2] No such file or directory: '/Users/varshh06/Desktop/CV-END-TO-END/03. ARTIFICIAL MAGIC SYTSTEM/mystic_motion_project/local_hand_features.csv'

In [58]:
counts = pose_df["label"].value_counts()
usable_classes = [label for label in TRAINED_LABELS if counts.get(label, 0) >= 5]
pose_df = pose_df[pose_df["label"].isin(usable_classes)].copy()

print("Usable class counts:", pose_df["label"].value_counts().to_dict())
if len(usable_classes) != 2:
    raise ValueError(
        "Both OPEN_PALM and TWO_FINGERS need at least 5 detected images. "
        f"Current counts: {counts.to_dict()}"
    )

NameError: name 'pose_df' is not defined

In [59]:
X = pose_df[FEATURE_COLUMNS].to_numpy(dtype=np.float32)
y = pose_df["label"].astype(str).to_numpy()
groups = pose_df["source_group"].astype(str).to_numpy()

NameError: name 'pose_df' is not defined

In [60]:
# Group split by source image prevents augmented/duplicate samples from leaking.
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
train_idx, test_idx = next(splitter.split(X, y, groups))
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
split_strategy = "25% group holdout by original image"

if len(set(y_train)) < 2 or len(set(y_test)) < 2:
    # Fallback for very small class counts; stratified split is preferable to a
    # one-class test partition. Each row still represents a different source file.
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=SEED, stratify=y
    )
    split_strategy = "stratified row holdout fallback"

print("Split strategy:", split_strategy)
print("Train samples:", len(y_train), "| Test samples:", len(y_test))
print("Train class counts:", dict(Counter(y_train)))
print("Test class counts:", dict(Counter(y_test)))

NameError: name 'X' is not defined

In [61]:
model = RandomForestClassifier(
    n_estimators=450,
    class_weight="balanced_subsample",
    min_samples_leaf=1,
    max_features="sqrt",
    random_state=SEED,
    n_jobs=-1
)

In [62]:

model.fit(X_train, y_train)
print("Training complete.")

NameError: name 'X_train' is not defined

In [63]:
predictions = model.predict(X_test)

NameError: name 'X_test' is not defined

In [64]:
test_accuracy = accuracy_score(y_test, predictions)
print("Split strategy:", split_strategy)
print("Train rows:", len(y_train), "| Test rows:", len(y_test))
print("Held-out accuracy (measured):", round(float(test_accuracy), 4))

NameError: name 'y_test' is not defined

In [65]:
print(classification_report(y_test, predictions, labels=usable_classes, zero_division=0))
ConfusionMatrixDisplay.from_predictions(
    y_test, predictions, labels=usable_classes, xticks_rotation=35
)
plt.title("MYSTIC MOTION — Local Hand Gesture Dataset")
plt.tight_layout()
plt.show()

NameError: name 'y_test' is not defined

In [66]:
model_bundle = {
    "model": model,
    "feature_columns": FEATURE_COLUMNS,
    "classes": list(model.classes_),
    "dataset_source": "shimaazizi/HandGesture-Recognizer dataset (locally downloaded)",
    "trained_gestures": ["OPEN_PALM", "TWO_FINGERS"],
    "point_method": "MediaPipe landmark rule (not supervised-trained)",
    "test_accuracy": float(test_accuracy),
    "normalization": "wrist-origin, max-XY-radius scale, 21x3",
    "confidence_threshold": 0.78,
}
joblib.dump(model_bundle, MODEL_PATH)
print("Saved model bundle:", MODEL_PATH.resolve())

AttributeError: 'RandomForestClassifier' object has no attribute 'classes_'

In [67]:
trained_bundle = joblib.load(MODEL_PATH)
live_pose_model = trained_bundle["model"]
print("Loaded trained model classes:", live_pose_model.classes_)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/varshh06/Desktop/CV-END-TO-END/03. ARTIFICIAL MAGIC SYTSTEM/mystic_motion_project/mystic_motion_gesture_model.joblib'

In [68]:
class LiveHandSmoother:
    def __init__(self, alpha=0.42, history_size=5):
        self.alpha = alpha
        self.points = {}
        self.poses = {}

    def update(self, hand, pose):
        key = hand["handedness"]
        current = hand["points"]
        if key in self.points:
            current = self.alpha * current + (1.0 - self.alpha) * self.points[key]
        self.points[key] = current
        hand["points"] = current
        history = self.poses.setdefault(key, deque(maxlen=5))
        history.append(pose)
        stable = Counter(history).most_common(1)[0][0]
        return hand, stable

    def retain(self, visible):
        for key in list(self.points):
            if key not in visible:
                self.points.pop(key, None)
                self.poses.pop(key, None)

In [69]:
def detect_point_rule(points):
    """Rule-based POINT detector using MediaPipe's 21 landmarks.

    It is not a trained classifier. It looks for an extended index finger
    while middle, ring and pinky remain folded.
    """
    states = finger_states(points)  # [thumb, index, middle, ring, pinky]
    index_extended = states[1]
    other_three_folded = not any(states[2:])
    return bool(index_extended and other_three_folded)

def predict_gesture(hand, model, threshold=CONFIDENCE_THRESHOLD):
    points = hand["points"]
    vector = normalized_features(points)
    if vector is None:
        return "UNKNOWN", 0.0

    # Detect POINT separately because the downloaded dataset has no pointing class.
    if detect_point_rule(points):
        return "POINT", 0.82

    probabilities = model.predict_proba(vector.reshape(1, -1))[0]
    order = np.argsort(probabilities)[::-1]
    index = int(order[0])
    confidence = float(probabilities[index])
    second = float(probabilities[order[1]]) if len(order) > 1 else 0.0

    # Confidence + margin help avoid forcing every unfamiliar pose into a class.
    if confidence < threshold or (confidence - second) < 0.12:
        return "UNKNOWN", confidence
    return str(model.classes_[index]), confidence

In [70]:
def hand_anchors(hand, width, height, gesture):
    p = hand["points"]
    palm = np.mean(p[[0, 5, 9, 13, 17], :2], axis=0)
    if gesture == "PINCH":
        return [((p[4, :2] + p[8, :2]) / 2, "pinch")]
    if gesture == "TWO_FINGERS":
        return [(p[8, :2], "index"), (p[12, :2], "middle"),
                ((p[8, :2] + p[12, :2]) / 2, "between")]
    if gesture == "POINT":
        return [(p[8, :2], "index"), (p[6, :2], "joint")]
    if gesture == "OPEN_PALM":
        return [(p[i, :2], "tip") for i in (4, 8, 12, 16, 20)] + [(palm, "palm")]
    if gesture == "FIST":
        return [(palm, "fist"), (p[0, :2], "wrist")]
    return [(palm, "palm")]

In [71]:
def px(point, width, height):
    return int(point[0] * width), int(point[1] * height)

def alpha_composite(frame, overlay, alpha=0.72):
    cv2.addWeighted(overlay, alpha, frame, 1.0 - alpha, 0, frame)

def glow_dot(layer, center, radius, color):
    cv2.circle(layer, center, radius * 3, color, -1, cv2.LINE_AA)
    cv2.circle(layer, center, radius, (245, 250, 255), -1, cv2.LINE_AA)

def draw_ring(layer, center, radius, color, t, count=3, tilt=0):
    cx, cy = center
    for i in range(count):
        r = max(8, int(radius * (0.62 + 0.22 * i) + 4 * math.sin(t * 2.4 + i)))
        angle = (t * (30 + i * 9) + i * 37 + tilt) % 360
        cv2.ellipse(layer, (cx, cy), (r, max(5, int(r * (0.42 + 0.12 * i)))),
                    int(angle), 0, 340, color, 2, cv2.LINE_AA)

In [72]:
def draw_arc(layer, a, b, color, t, branches=2):
    ax, ay = a
    bx, by = b
    mid = np.array([(ax + bx) / 2, (ay + by) / 2], dtype=float)
    direction = np.array([-(by - ay), bx - ax], dtype=float)
    norm = np.linalg.norm(direction) + 1e-6
    direction /= norm
    pts = [a]
    for j in range(1, 7):
        base = np.array(a, dtype=float) + (np.array(b, dtype=float) - a) * j / 7
        jitter = math.sin(t * 18 + j * 4.1) * (5 + branches * 2)
        q = base + direction * jitter
        pts.append(tuple(q.astype(int)))
    pts.append(b)
    cv2.polylines(layer, [np.asarray(pts, np.int32)], False, color, 2, cv2.LINE_AA)
    cv2.polylines(layer, [np.asarray(pts, np.int32)], False, (245, 250, 255), 1, cv2.LINE_AA)

In [73]:
def draw_lightning(layer, hand, gesture, t, width, height):
    p = hand["points"]
    if gesture == "POINT":
        path = [px(p[i, :2], width, height) for i in (5, 6, 7, 8)]
        for a, b in zip(path[:-1], path[1:]):
            draw_arc(layer, a, b, (255, 220, 105), t, 3)
        draw_ring(layer, path[-1], 23, (255, 235, 140), t, 3)
        for k in range(3):
            orbit = (t * 3.4 + k * 2.1)
            q = (int(path[-1][0] + 15 * math.cos(orbit)), int(path[-1][1] + 15 * math.sin(orbit)))
            draw_arc(layer, path[-1], q, (255, 255, 220), t + k, 2)
    elif gesture == "TWO_FINGERS":
        a, b = px(p[8, :2], width, height), px(p[12, :2], width, height)
        draw_arc(layer, a, b, (255, 245, 180), t, 4)
        draw_ring(layer, a, 20, (255, 220, 100), t, 2)
        draw_ring(layer, b, 20, (255, 220, 100), -t, 2)
    elif gesture == "OPEN_PALM":
        tips = [px(p[i, :2], width, height) for i in (4, 8, 12, 16, 20)]
        palm = px(np.mean(p[[0, 5, 9, 13, 17], :2], axis=0), width, height)
        draw_ring(layer, palm, 48, (255, 215, 85), t, 4)
        for tip in tips:
            draw_arc(layer, palm, tip, (255, 245, 190), t, 4)
    elif gesture == "FIST":
        palm = px(np.mean(p[[0, 5, 9, 13, 17], :2], axis=0), width, height)
        draw_ring(layer, palm, 43, (255, 215, 80), t, 5)
        for i in range(12):
            angle = t * 2.8 + i * math.tau / 12
            q = (int(palm[0] + 38 * math.cos(angle)), int(palm[1] + 30 * math.sin(angle)))
            draw_arc(layer, palm, q, (255, 255, 225), t + i, 2)
    elif gesture == "PINCH":
        center = px((p[4, :2] + p[8, :2]) / 2, width, height)
        draw_ring(layer, center, 26, (255, 230, 105), t, 4)
        for i in range(16):
            angle = t * 2.5 + i * math.tau / 16
            q = (int(center[0] + 18 * math.cos(angle)), int(center[1] + 18 * math.sin(angle)))
            glow_dot(layer, q, 2, (255, 255, 245))

In [74]:
def draw_character_gesture(frame, character, gesture, hand, t, width, height):
    """Character-specific abstract magic VFX anchored to detected hand landmarks."""
    overlay = frame.copy()
    p = hand["points"]
    palm = px(np.mean(p[[0, 5, 9, 13, 17], :2], axis=0), width, height)
    color = PALETTES[character]
    anchors = hand_anchors(hand, width, height, gesture)
    points = [(px(a, width, height), kind) for a, kind in anchors]

    if character == "THOR":
        draw_lightning(overlay, hand, gesture, t, width, height)

    for center, kind in points:
        x, y = center

        if character == "DOCTOR STRANGE":
            # Layered orange sorcerer sigils and rotating runic orbit.
            draw_ring(overlay, center, 60 if gesture == "OPEN_PALM" else 43, color, t, 6)
            for i in range(12):
                a = t * 0.8 + i * math.tau / 12
                q = (int(x + 52 * math.cos(a)), int(y + 52 * math.sin(a)))
                cv2.line(overlay, center, q, (95, 205, 255), 1, cv2.LINE_AA)
                cv2.circle(overlay, q, 3, (190, 235, 255), -1, cv2.LINE_AA)

        elif character == "SCARLET WITCH":
            # Red chaos-energy halo with tendrils.
            draw_ring(overlay, center, 52, color, -t * 1.4, 5)
            for i in range(10):
                a = t * 1.7 + i * math.tau / 10
                q = (int(x + 40 * math.cos(a)), int(y + 32 * math.sin(a)))
                draw_arc(overlay, center, q, (90, 70, 255), t + i, 3)
            glow_dot(overlay, center, 8, (100, 100, 255))

        elif character == "AGATHA HARKNESS":
            # Violet witchcraft: orbiting wisps and branching arcs.
            draw_ring(overlay, center, 46, color, t * 1.3, 4)
            for i in range(7):
                a = -t + i * math.tau / 7
                q = (int(x + 54 * math.cos(a)), int(y + 42 * math.sin(a)))
                draw_arc(overlay, center, q, (220, 130, 255), t + i, 3)
                glow_dot(overlay, q, 3, color)

        elif character == "LOKI":
            # Emerald illusion spiral.
            for i in range(3):
                spiral = []
                for j in range(36):
                    a = j * math.tau / 35 + t * (1 if i % 2 else -1) + i * 2.1
                    r = 10 + j * 1.45
                    spiral.append((int(x + math.cos(a) * r), int(y + math.sin(a) * r * 0.62)))
                cv2.polylines(overlay, [np.asarray(spiral, np.int32)], False, color, 2, cv2.LINE_AA)
            glow_dot(overlay, center, 5, (170, 255, 190))

        elif character == "THOR":
            # Extra orbiting sparks around the lightning effect.
            for i in range(3):
                a = t * 2 + i * math.tau / 3
                q = (int(x + 42 * math.cos(a)), int(y + 30 * math.sin(a)))
                draw_arc(overlay, center, q, (255, 250, 190), t + i, 4)

        elif character == "HELA":
            # Dark emerald blade-like rays.
            draw_ring(overlay, center, 45, color, -t, 3)
            for i in range(8):
                a = t * 0.45 + i * math.tau / 8
                inner = (int(x + 13 * math.cos(a)), int(y + 13 * math.sin(a)))
                outer = (int(x + 75 * math.cos(a)), int(y + 55 * math.sin(a)))
                cv2.line(overlay, inner, outer, color, 3, cv2.LINE_AA)
                cv2.line(overlay, inner, outer, (170, 255, 190), 1, cv2.LINE_AA)

        elif character == "WONG":
            # Dense golden-orange mandala shield.
            draw_ring(overlay, center, 50, color, t * 0.65, 7)
            for i in range(8):
                a = t * 0.35 + i * math.tau / 8
                q = (int(x + 42 * math.cos(a)), int(y + 42 * math.sin(a)))
                cv2.circle(overlay, q, 6, (130, 210, 255), 1, cv2.LINE_AA)
                cv2.line(overlay, center, q, (110, 185, 255), 1, cv2.LINE_AA)

        elif character == "WICCAN":
            # Blue-violet star field and arcane energy.
            draw_ring(overlay, center, 48, color, t, 5)
            for i in range(6):
                a = -t * 0.9 + i * math.tau / 6
                q = (int(x + 58 * math.cos(a)), int(y + 58 * math.sin(a)))
                draw_arc(overlay, center, q, (255, 170, 235), t + i, 2)
                glow_dot(overlay, q, 3, color)

        elif character == "THE ANCIENT ONE":
            # Calm golden concentric time rings.
            draw_ring(overlay, center, 62, color, t * 0.35, 7)
            for i in range(16):
                a = -t * 0.25 + i * math.tau / 16
                q = (int(x + 55 * math.cos(a)), int(y + 55 * math.sin(a)))
                cv2.circle(overlay, q, 2, (230, 250, 170), -1, cv2.LINE_AA)

    # Gesture-responsive central aura.
    if gesture == "OPEN_PALM":
        draw_ring(overlay, palm, 70, color, t, 3)
    elif gesture == "TWO_FINGERS":
        a = px(p[8, :2], width, height)
        b = px(p[12, :2], width, height)
        draw_arc(overlay, a, b, color, t, 3)
        draw_ring(overlay, ((a[0] + b[0]) // 2, (a[1] + b[1]) // 2), 28, color, t, 3)
    elif gesture == "POINT":
        tip = px(p[8, :2], width, height)
        draw_ring(overlay, tip, 28, color, t, 3)
        draw_arc(overlay, palm, tip, color, t, 4)

    alpha_composite(frame, overlay, 0.68)

In [75]:
def run_mystic_motion(camera_index=0, confidence_threshold=CONFIDENCE_THRESHOLD):
    cap = cv2.VideoCapture(camera_index)
    if not cap.isOpened():
        cap.release()
        raise RuntimeError("Cannot open webcam. Check camera permissions or try camera_index=1.")

    tracker = HandTracker(max_hands=2)
    smoother = LiveHandSmoother()
    character = "DOCTOR STRANGE"
    show_skeleton = True
    show_help = True
    start_time = time.time()
    frame_failures = 0

    print("MYSTIC MOTION is running.")
    print("Keys 1–9: select character | Q: quit | S: landmarks | T: help")

    try:
        while True:
            ok, frame = cap.read()
            if not ok or frame is None:
                frame_failures += 1
                if frame_failures >= 10:
                    print("Camera repeatedly failed to provide frames; stopping.")
                    break
                time.sleep(0.03)
                continue

            frame_failures = 0
            frame = cv2.flip(frame, 1)
            height, width = frame.shape[:2]
            elapsed = time.time() - start_time
            hands = tracker.process(frame)
            predictions = []
            visible = set()

            for hand in hands:
                key_name = hand["handedness"]
                visible.add(key_name)
                raw_pose, confidence = predict_gesture(hand, live_pose_model, confidence_threshold)
                hand, stable_pose = smoother.update(hand, raw_pose)
                predictions.append((hand, stable_pose, confidence))
                if show_skeleton:
                    mp_draw.draw_landmarks(frame, hand["landmarks"], mp_hands.HAND_CONNECTIONS)

            smoother.retain(visible)

            active, midpoint, radius, separation = two_hand_opposed_circle(hands)
            if active:
                center = (int(midpoint[0] * width), int(midpoint[1] * height))
                draw_two_hand_circle(frame, center, radius, character, elapsed)
                cv2.putText(frame, "TWO-HAND PORTAL", (center[0] - 105, center[1] + radius + 24),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.62, (245, 245, 255), 2)
            else:
                for hand, gesture, confidence in predictions:
                    if gesture in POSE_LABELS:
                        draw_character_gesture(frame, character, gesture, hand, elapsed, width, height)

            primary = predictions[0] if predictions else None
            gesture_text = primary[1] if primary else "NO HAND"
            conf_text = primary[2] if primary else 0.0

            cv2.rectangle(frame, (0, 0), (width, 100), (12, 12, 25), -1)
            cv2.putText(frame, f"MYSTIC MOTION | {character}", (14, 31),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.72, (150, 225, 255), 2)
            cv2.putText(frame, f"Gesture: {gesture_text} | Score: {conf_text:.2f} | Hands: {len(hands)}",
                        (14, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.53, (255, 255, 255), 1)

            if show_help:
                help_text = "1 Strange 2 Wanda 3 Agatha 4 Loki 5 Thor 6 Hela 7 Wong 8 Wiccan 9 Ancient One"
                cv2.putText(frame, help_text, (14, height - 18),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.43, (180, 230, 255), 1)

            cv2.imshow("MYSTIC MOTION — Character Magic AR", frame)
            key = cv2.waitKey(1) & 0xFF

            if key == ord("q"):
                break
            elif key == ord("s"):
                show_skeleton = not show_skeleton
            elif key == ord("t"):
                show_help = not show_help
            elif 49 <= key <= 57:
                character = CHARACTERS[chr(key)]
                print("Selected character:", character)

    finally:
        cap.release()
        tracker.close()
        cv2.destroyAllWindows()


# Launch the live AR experience after all training and effect cells have run.
run_mystic_motion(camera_index=0)

MYSTIC MOTION is running.
Keys 1–9: select character | Q: quit | S: landmarks | T: help


I0000 00:00:1793956511.365552  927830 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M3
W0000 00:00:1793956511.384027 1017485 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1793956511.401319 1017483 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


NameError: name 'two_hand_opposed_circle' is not defined

In [76]:
def draw_two_hand_circle(frame, center, radius, character, t):
    """Two open hands control a large character-colored portal/energy sphere."""
    overlay = frame.copy()
    color = PALETTES[character]
    c = tuple(map(int, center))
    for i in range(6):
        r = int(radius * (0.42 + i * 0.105) + 5 * math.sin(t * 1.8 + i))
        cv2.circle(overlay, c, max(8, r), color if i % 2 else (235, 245, 255), 2, cv2.LINE_AA)
    for i in range(32):
        a = t * (0.45 if i % 2 else -0.55) + i * math.tau / 32
        inner = radius * 0.35
        outer = radius * (0.82 if i % 2 else 0.68)
        p1 = (int(c[0] + inner * math.cos(a)), int(c[1] + inner * math.sin(a)))
        p2 = (int(c[0] + outer * math.cos(a)), int(c[1] + outer * math.sin(a)))
        cv2.line(overlay, p1, p2, color, 2, cv2.LINE_AA)
        cv2.circle(overlay, p2, 3, (245, 250, 255), -1, cv2.LINE_AA)
    cv2.circle(overlay, c, max(10, int(radius * 0.20)), color, 2, cv2.LINE_AA)
    alpha_composite(frame, overlay, 0.70)

print("Nine character effect styles and two-hand portal effect are ready.")

Nine character effect styles and two-hand portal effect are ready.


In [77]:
def two_hand_opposed_circle(hands):
    if len(hands) < 2:
        return False, None, None, None

    first, second = hands[0], hands[1]
    p1, p2 = first["points"], second["points"]
    v1 = p1[9, :2] - p1[0, :2]
    v2 = p2[9, :2] - p2[0, :2]
    denom = np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-8
    cosine = float(np.dot(v1, v2) / denom)

    c1 = np.mean(p1[[0, 5, 9, 13, 17], :2], axis=0)
    c2 = np.mean(p2[[0, 5, 9, 13, 17], :2], axis=0)
    separation = float(np.linalg.norm(c1 - c2))
    facing = cosine < -0.20
    open_pair = all(sum(finger_states(h["points"])) >= 4 for h in (first, second))
    active = facing and open_pair and 0.12 < separation < 0.90

    return active, (c1 + c2) / 2.0, max(70, int(separation * 460)), separation